# 03 — Inversi dispersi menjadi profil Vs

**Alur:** `pick dispersi → model lapisan 1D → kurva dispersi teoretis → optimasi dan QC model`.

**Input:** pick dan status review dari notebook 02, ditambah batas ketebalan, Vs, sifat elastik, dan pengaturan optimasi pada sel di bawah.

**Proses:** hitung kecepatan fase Rayleigh untuk setiap model lapisan, lalu minimalkan selisihnya terhadap pick teramati. Beberapa run memperlihatkan apakah solusi sensitif terhadap titik awal; kecocokan numerik tetap harus dinilai bersama mutu input.

**Output:** profil kandidat, kurva fit, dan status inversi di `outputs/<site_id>/03/`. Bila pick belum memenuhi gerbang review, hasil berada di `preview/`. Notebook 04 memakai profil ini untuk pemeriksaan terhadap HVSR.


**Acuan MAM:** [Hayashi et al. (2022)](https://doi.org/10.1007/s10950-021-10051-y), terutama §6–7. Komponen HVSR/QC SESAME tetap memakai acuan khususnya.


## Input pengguna — asumsi lapisan dan anggaran inversi

`finite_layer_thickness_bounds_m` berisi batas `(min, max)` ketebalan lapisan hingga; `layer_vs_bounds_m_s` harus mempunyai satu baris lebih banyak karena baris terakhir adalah halfspace. `preview_*` dipakai saat QC input belum lulus, `final_*` saat pick final sudah tersedia. Nilai ini tersimpan bersama keluaran agar inversi dapat diulang.

In [ ]:
SITE_ID = "solo_pilot"
INVERSION_PARAMETERS = {
    "finite_layer_thickness_bounds_m": [[2.0, 8.0], [8.0, 20.0]],
    "layer_vs_bounds_m_s": [[150.0, 550.0], [200.0, 850.0], [250.0, 1200.0]],
    "initial_vs_phase_velocity_factor": 1.0,  # project initialization assumption, not measured Vs
    "poisson_ratio_assumed": 0.30,
    "density_assumed_g_cm3": 2.0,
    "minimum_accepted_picks": 8,
    "preview_runs": 3, "preview_maxiter": 60, "preview_popsize_multiplier": 6,
    "final_runs": 3, "final_maxiter": 60, "final_popsize_multiplier": 8,
    "seeds": [2026, 2027, 2028],
}

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
from datetime import UTC, datetime
from pathlib import Path

import disba
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from disba import DispersionError
from scipy.optimize import differential_evolution

from mhvsr_vs30.mam.inversion import forward_rayleigh_phase
from mhvsr_vs30.mam.picking import review_context_hash
from mhvsr_vs30.mam.wavelength import (wavelength_guides, wavelength_diagnostics,
    depth_guidelines, initial_model_from_wavelength)

ROOT=Path.cwd()
CONFIG={'site_id':SITE_ID}
PARAM=INVERSION_PARAMETERS
SOURCE=ROOT/'outputs'/CONFIG['site_id']/'02'
SITE_INPUTS=ROOT/'outputs'/CONFIG['site_id']/'01'/'site_inputs.json'
OUT=ROOT/'outputs'/CONFIG['site_id']/'03'
OUT.mkdir(parents=True,exist_ok=True)
FINAL_INPUT=SOURCE/'dispersion_final.csv'
AUTO_INPUT=SOURCE/'dispersion_auto.csv'
QC_INPUT=SOURCE/'dispersion_qc.json'
assert FINAL_INPUT.is_file() and AUTO_INPUT.is_file() and QC_INPUT.is_file()
assert SITE_INPUTS.is_file(), f'Run notebook 01 first: {SITE_INPUTS} missing'
print('Inputs:',FINAL_INPUT,AUTO_INPUT,QC_INPUT)
(OUT/'inversion_status.json').write_text(json.dumps({'run_state':'running',
    'method_revision':'hayashi_2022_v1'}),encoding='utf-8')


## 1. Gerbang input dan mode eksekusi

`accepted_for_inversion=True` serta kecepatan final terisi diperlukan untuk jalur final. Metadata notebook 02 juga harus menyatakan geometri, drift jam, dan mode sudah diverifikasi. Bila syarat ini belum terpenuhi, notebook menggunakan hanya kandidat `provisional` untuk preview dan tidak mengubah CSV dispersi final.


In [ ]:
final_df=pd.read_csv(FINAL_INPUT)
auto_df=pd.read_csv(AUTO_INPUT)
upstream=json.loads(QC_INPUT.read_text(encoding='utf-8'))
if upstream.get('run_state')!='complete' or upstream.get('method_revision')!='hayashi_2022_v1':
    raise ValueError('Notebook 02 must complete with the current method before inversion')
for artifact,key in ((AUTO_INPUT,'dispersion_auto_sha256'),(FINAL_INPUT,'dispersion_final_sha256')):
    if hashlib.sha256(artifact.read_bytes()).hexdigest()!=upstream.get(key):
        raise ValueError('Dispersion picks changed since QC: rerun notebook 02')
review_context=review_context_hash(SITE_INPUTS,SOURCE/'input_inventory.csv',AUTO_INPUT)
if upstream.get('review_context_sha256')!=review_context:
    raise ValueError('Dispersion review context changed; rerun notebook 02 before inversion')
accepted=final_df.accepted_for_inversion.astype(str).str.lower().eq('true')
if (accepted & final_df.automatic_status.eq('rejected_numeric')).any():
    raise ValueError('Selected final pick failed numerical QC')
reviewed=final_df.loc[accepted].copy()
required_flags=('geometry_verified','physical_clock_drift_verified','wavefield_mode_verified','wavelength_range_reviewed')
flags={name:bool(upstream.get(name,False)) for name in required_flags}
final_ready=(len(reviewed)>=PARAM['minimum_accepted_picks'] and all(flags.values()))
mode='final' if final_ready else 'preview'
if mode=='final':
    selection=reviewed[['period_s','frequency_hz','velocity_final_m_s']].rename(
        columns={'velocity_final_m_s':'observed_velocity_m_s'})
else:
    selection=auto_df.loc[auto_df.automatic_status.eq('provisional'),
        ['period_s','frequency_hz','velocity_auto_m_s']].rename(
        columns={'velocity_auto_m_s':'observed_velocity_m_s'})
selection=selection.sort_values('period_s').reset_index(drop=True)
if (not np.isfinite(selection[['period_s','frequency_hz','observed_velocity_m_s']]).all().all()
    or (selection[['period_s','frequency_hz','observed_velocity_m_s']]<=0).any().any()
    or not np.allclose(selection.period_s*selection.frequency_hz,1.0)):
    raise ValueError('Dispersion frequencies, periods and velocities must be finite, positive and consistent')
diagnostics=wavelength_diagnostics(selection.frequency_hz.to_numpy(),
    selection.observed_velocity_m_s.to_numpy(),upstream['maximum_receiver_spacing_m'],
    maximum_wavelength_ratio=upstream['processing_parameters'].get('maximum_wavelength_ratio'))
if not diagnostics['within_wavelength_limit'].all():
    raise ValueError('Selected picks exceed the project wavelength limit; review notebook 02')
depth_summary=depth_guidelines(selection.frequency_hz.to_numpy(),selection.observed_velocity_m_s.to_numpy())
assert len(selection)>=PARAM['minimum_accepted_picks'], 'not enough picks even for preview'
assert (selection.period_s>0).all() and (selection.observed_velocity_m_s>0).all()
assert selection.period_s.is_monotonic_increasing and not selection.period_s.duplicated().any()
period=selection.period_s.to_numpy(dtype=float)
observed=selection.observed_velocity_m_s.to_numpy(dtype=float)
RUN_OUT=OUT/('preview' if mode=='preview' else 'final')
RUN_OUT.mkdir(parents=True,exist_ok=True)
print('Mode:',mode,'| picks:',len(selection),'| final flags:',flags)

pd.DataFrame(diagnostics).to_csv(RUN_OUT/'wavelength_diagnostics.csv',index=False)


## 2. Panduan panjang gelombang dan model awal

Hayashi §7.2/Figure 5 memakai pasangan `(z_guide=λ/3, c)` dengan `λ=c/f` sebagai panduan. Ini bukan profil Vs terukur dan tidak memakai faktor 1,09 milik pendekatan Zor.

Parameterisasi proyek mempertahankan ketebalan awal pada titik tengah bounds. Kecepatan fase panduan diinterpolasi terhadap kedalaman pada tengah tiap lapisan dan puncak halfspace, lalu dikalikan `initial_vs_phase_velocity_factor` (default 1: asumsi Vs awal≈c). Kedalaman di luar panduan memakai nilai ujung dan ditandai tidak terikat; nilai Vs dipotong ke bounds dan pemotongannya dicatat. Tidak ada asumsi Vs meningkat monoton.

Vektor ini diberikan sebagai `x0` ke setiap run differential evolution, menggantikan satu anggota populasi awal; anggota lain tetap bervariasi menurut seed. Bounds tetap asumsi pengguna dan tidak otomatis menjadi hasil pengukuran. `wavelength_guides.csv` dan `initial_model_diagnostics.csv` memperlihatkan perbedaannya.


In [ ]:
thickness_bounds=[tuple(map(float,b)) for b in PARAM['finite_layer_thickness_bounds_m']]
vs_bounds=[tuple(map(float,b)) for b in PARAM['layer_vs_bounds_m_s']]
assert len(vs_bounds)==len(thickness_bounds)+1
bounds=thickness_bounds+vs_bounds
assert all(0<low<high for low,high in bounds)
n_finite=len(thickness_bounds)
poisson=float(PARAM['poisson_ratio_assumed'])
density=float(PARAM['density_assumed_g_cm3'])
guides=wavelength_guides(selection.frequency_hz.to_numpy(),observed)
pd.DataFrame(guides).to_csv(RUN_OUT/'wavelength_guides.csv',index=False)
initial_info=initial_model_from_wavelength(selection.frequency_hz.to_numpy(),observed,
    thickness_bounds,vs_bounds,velocity_factor=PARAM['initial_vs_phase_velocity_factor'])
initial=initial_info['parameters']
initial_method=('wavelength_endpoint_assumption' if initial_info['outside_guide_depth'].all()
    else 'wavelength_interpolation_project_vs_approximation')
initial_label=('Initial: endpoint assumptions' if initial_info['outside_guide_depth'].all()
    else 'Initial: wavelength interpolation')
pd.DataFrame({key:value for key,value in initial_info.items() if key!='parameters'}).to_csv(
    RUN_OUT/'initial_model_diagnostics.csv',index=False)

def predict(parameters):
    return forward_rayleigh_phase(period,parameters[:n_finite],parameters[n_finite:],
                                  poisson=poisson,density_g_cm3=density)

def objective(parameters):
    try:
        calculated=predict(parameters)
    except (DispersionError,RuntimeError,ValueError):
        return 1e6
    if not np.isfinite(calculated).all():
        return 1e6
    return float(np.sqrt(np.mean((calculated-observed)**2)))

def layer_table(parameters,model_label):
    thickness=np.asarray(parameters[:n_finite],dtype=float)
    vs=np.asarray(parameters[n_finite:],dtype=float)
    vp=vs*np.sqrt(2*(1-poisson)/(1-2*poisson))
    tops=np.r_[0,np.cumsum(thickness)]
    bottoms=np.r_[np.cumsum(thickness),np.nan]
    return pd.DataFrame({'model':model_label,'layer_index':np.arange(1,len(vs)+1),
        'top_depth_m':tops,'bottom_depth_m':bottoms,
        'thickness_m':np.r_[thickness,np.nan],'vs_m_s':vs,'vp_m_s':vp,
        'density_g_cm3':density,'poisson_assumed':poisson})

initial_df=layer_table(initial,initial_method)
initial_df.to_csv(RUN_OUT/'vs_initial.csv',index=False)
print(initial_df[['layer_index','top_depth_m','bottom_depth_m','vs_m_s']].to_string(index=False))

print('Initial samples outside guide depth:',int(initial_info['outside_guide_depth'].sum()),'of',len(initial_info['outside_guide_depth']),'; endpoint values are assumptions')


## 3. Inversi beberapa seed dan ensemble hasil pengoptimasi

Setiap seed menghasilkan populasi akhir, bukan posterior statistik. Metrik adalah RMSE kecepatan fasa dalam m/s tanpa bobot. Jumlah run dan iterasi mengikuti sel input di atas. Mengubah anggaran optimasi memerlukan menjalankan ulang notebook ini. Bila JIT pertama kali lambat, jalankan sel ini sampai selesai dan simpan metadata sebelum menilai model.
Hayashi menyarankan least squares fundamental mode sebagai pilihan awal. Differential evolution + disba dipertahankan sebagai pilihan implementasi proyek. Forward masih Rayleigh R0: optimizer global tidak membuatnya multimode. Higher modes memerlukan identifikasi observasi dan forward yang sesuai. MMSPAC belum tersedia.


In [ ]:
n_runs=int(PARAM['preview_runs'] if mode=='preview' else PARAM['final_runs'])
maxiter=int(PARAM['preview_maxiter'] if mode=='preview' else PARAM['final_maxiter'])
popsize=int(PARAM['preview_popsize_multiplier'] if mode=='preview' else PARAM['final_popsize_multiplier'])
seeds=list(PARAM['seeds'])[:n_runs]
assert len(seeds)==n_runs
run_results=[]
members=[]
for run_index,seed in enumerate(seeds,1):
    result=differential_evolution(objective,bounds,maxiter=maxiter,popsize=popsize,
        x0=initial.copy(),rng=np.random.default_rng(seed),workers=1,polish=False,tol=0.01)
    run_results.append((run_index,seed,result))
    for member_index,(parameters,misfit) in enumerate(
            zip(result.population,result.population_energies,strict=True)):
        members.append({'run':run_index,'seed':seed,'member':member_index,
            'rmse_m_s':float(misfit),
            **{f'thickness_{i+1}_m':float(parameters[i]) for i in range(n_finite)},
            **{f'vs_layer_{i+1}_m_s':float(v) for i,v in enumerate(parameters[n_finite:])}})
    print('Run',run_index,'seed',seed,'best RMSE m/s:',float(result.fun),
          '| evaluations:',result.nfev,'| converged:',bool(result.success))
ensemble_df=pd.DataFrame(members).sort_values('rmse_m_s').reset_index(drop=True)
ensemble_df.to_csv(RUN_OUT/'vs_dispersion_ensemble.csv',index=False)
best_run,best_seed,best_result=min(run_results,key=lambda item:item[2].fun)
best=np.asarray(best_result.x,dtype=float)
best_df=layer_table(best,'preview_exploratory' if mode=='preview' else 'inverted_final_input')
best_df.to_csv(RUN_OUT/'vs_dispersion_best.csv',index=False)
calculated=predict(best)
fit_df=selection.copy()
fit_df['predicted_velocity_m_s']=calculated
fit_df['residual_m_s']=calculated-observed
fit_df.to_csv(RUN_OUT/'dispersion_fit.csv',index=False)
print('Best run:',best_run,'| RMSE m/s:',float(best_result.fun))
print(best_df[['layer_index','top_depth_m','bottom_depth_m','vs_m_s']].to_string(index=False))


## 4. Fit, status dan batas interpretasi

Grafik model menampilkan best run dan semua anggota populasi akhir yang cukup baik untuk melihat kemungkinan non-keunikan. Sebaran populasi mencerminkan pencarian optimisasi dalam batas yang dipilih, **bukan** ketidakpastian probabilistik. Preview tidak menulis Vs30 karena cakupan frekuensi tinggi, posisi array, dan mode belum memastikan resolusi kedalaman 30 m.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,5))
axes[0].plot(observed,period,'ko',ms=4,label='Input candidates' if mode=='preview' else 'Accepted input')
axes[0].plot(calculated,period,'r-',lw=1.5,label='Forward best')
axes[0].set(xlabel='Rayleigh phase velocity (m/s)',ylabel='Period (s)',
            title='Dispersion fit (RMSE %.1f m/s)'%best_result.fun)
axes[0].grid(alpha=0.2); axes[0].legend(fontsize=8)

def step_model(ax,parameters,**kwargs):
    th=np.asarray(parameters[:n_finite]); vs=np.asarray(parameters[n_finite:])
    edges=np.r_[0,np.cumsum(th),40.0]
    ax.step(np.r_[vs,vs[-1]],edges,where='pre',**kwargs)

cutoff=float(ensemble_df.rmse_m_s.min())+max(10.0,0.25*float(ensemble_df.rmse_m_s.min()))
shown=0
for _,row in ensemble_df.loc[ensemble_df.rmse_m_s<=cutoff].head(80).iterrows():
    params=np.r_[[row[f'thickness_{i+1}_m'] for i in range(n_finite)],
                 [row[f'vs_layer_{i+1}_m_s'] for i in range(n_finite+1)]]
    step_model(axes[1],params,color='0.75',lw=0.6,alpha=0.4)
    shown+=1
step_model(axes[1],initial,color='tab:blue',lw=1,linestyle='--',label=initial_label)
step_model(axes[1],best,color='tab:red',lw=2,label='Best model')
axes[1].invert_yaxis(); axes[1].set(xlabel='Vs (m/s)',ylabel='Depth (m)',
    title=f'Model ensemble (shown {shown})',ylim=(40,0))
axes[1].grid(alpha=0.2); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(RUN_OUT/'inversion_qc.png',dpi=160); plt.show()

fig,ax=plt.subplots(figsize=(6,5))
ax.scatter(guides['phase_velocity_m_s'],guides['guide_depth_m'],s=18,label='c at wavelength/3 (not measured Vs)')
step_model(ax,initial,color='tab:blue',label=initial_label)
ax.invert_yaxis(); ax.set(xlabel='Phase velocity / initial Vs (m/s)',ylabel='Depth (m)',title='Wavelength guidance and layer assumption')
ax.legend(fontsize=7); ax.grid(alpha=0.2); fig.tight_layout()
fig.savefig(RUN_OUT/'wavelength_initial_qc.png',dpi=160); plt.show()

status={
    'run_state':'complete','method_revision':'hayashi_2022_v1',
    'primary_reference_doi':'10.1007/s10950-021-10051-y',
    'initial_model_method':initial_method,
    'initial_guide_supported_samples':int((~initial_info['outside_guide_depth']).sum()),
    'initial_model_used_by_optimizer':'x0_replaces_one_population_member_each_run',
    'initial_vs_phase_velocity_factor':PARAM['initial_vs_phase_velocity_factor'],
    'initial_outside_guide_depth':initial_info['outside_guide_depth'].tolist(),
    'initial_clipped_to_bounds':initial_info['clipped_to_bounds'].tolist(),
    'depth_guidelines_m':depth_summary,'depth_guidelines_input_level':mode,
    'depth_guidelines_are_resolution_proof':False,
    'model_sha256':hashlib.sha256((RUN_OUT/'vs_dispersion_best.csv').read_bytes()).hexdigest(),
    'fit_sha256':hashlib.sha256((RUN_OUT/'dispersion_fit.csv').read_bytes()).hexdigest(),
    'site_id':CONFIG['site_id'],'created_at_utc':datetime.now(UTC).isoformat(),
    'mode':mode,'input_level':'provisional_automatic_candidates' if mode=='preview' else 'reviewed_dispersion',
    'source_csv':(AUTO_INPUT if mode=='preview' else FINAL_INPUT).relative_to(ROOT).as_posix(),
    'source_sha256':hashlib.sha256((AUTO_INPUT if mode=='preview' else FINAL_INPUT).read_bytes()).hexdigest(),
    'upstream_qc_sha256':hashlib.sha256(QC_INPUT.read_bytes()).hexdigest(),
    'configuration_sha256':hashlib.sha256(json.dumps(PARAM,sort_keys=True).encode()).hexdigest(),
    'final_input_picks':len(reviewed),'input_picks_used':len(selection),
    'upstream_verification_flags':flags,'layer_bounds':bounds,
    'poisson_ratio_assumed':poisson,'density_assumed_g_cm3':density,
    'wave_assumed':'rayleigh','mode_assumed':0,'misfit':'unweighted_rmse_m_s',
    'optimizer':'scipy_differential_evolution','seeds':seeds,'maxiter':maxiter,
    'popsize_multiplier':popsize,'run_best_misfit_m_s':[float(r.fun) for _,_,r in run_results],
    'run_converged':[bool(r.success) for _,_,r in run_results],
    'run_function_evaluations':[int(r.nfev) for _,_,r in run_results],
    'best_run':best_run,'best_seed':best_seed,'best_rmse_m_s':float(best_result.fun),
    'ensemble_interpretation':'optimizer_population_not_statistical_interval',
    'vs30_reported':False,'vs30_reason':'depth_30m_not_demonstrably_resolved',
    'reviewed_input_inversion':mode=='final',
    'final_vs_profile':False,'fit_acceptance_review_pending':True,
    'python_version':platform.python_version(),'numpy_version':np.__version__,
    'scipy_version':scipy.__version__,'disba_version':disba.__version__,
}
(OUT/'inversion_status.json').write_text(json.dumps(status,indent=2),encoding='utf-8')
print('Saved:',RUN_OUT,'| final Vs profile:',status['final_vs_profile'],
      '| Vs30 reported:',status['vs30_reported'])